In [ ]:
import urllib.request
import zipfile
from functools import partial
import os
import sqlalchemy
from sqlalchemy import create_engine, func, desc
import pandas as pd

In [ ]:
import urllib.request
import zipfile
import os

chinook_url = 'http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip'
if not os.path.exists('chinook.db'):
    print('Downloading chinook.zip...')
    urllib.request.urlretrieve(chinook_url, 'chinook.zip')
    
    print('Extracting database...')
    with zipfile.ZipFile('chinook.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    
    print('✅ Database extracted successfully!')
else:
    print('✅ Database already exists!')

#### Exercise 1: Open the Database


In [ ]:
engine = create_engine('sqlite:///chinook.db')
cur = engine.connect()

# Uing reflection to map existing database structure
metadata = sqlalchemy.MetaData()
metadata.reflect(engine)

# Extract classes from the chinook database
from sqlalchemy.ext.automap import automap_base

# Produce a set of mappings from this MetaData
Base = automap_base(metadata=metadata)
Base.prepare()

# Prepare an ORM session
from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind=engine)
session = Session()

# Get table classes
Track = Base.classes.tracks
Album = Base.classes.albums
Artist = Base.classes.artists
InvoiceItem = Base.classes.invoice_items

print("✅ Database connected and ORM ready!")

### Exercise 2 : Table Names

In [ ]:
# Used metadata.tables.keys() to get all table names
table_names = list(metadata.tables.keys())
print("All table names in the Chinook database:")
for i, table in enumerate(table_names, 1):
    print(f"{i:2d}. {table}")
print(f"\nTotal tables: {len(table_names)}")

### Exercise3 : First three tracks

In [ ]:
first_three_tracks = session.query(Track).limit(3).all()

print("First 3 tracks in the tracks table:")
for i, track in enumerate(first_three_tracks, 1):
    print(f"{i}. ID: {track.TrackId}, Name: '{track.Name}', Composer: {track.Composer}")

# display using pandas
query = session.query(Track).limit(3)
df_tracks = pd.read_sql(query.statement, engine)
print("\nUsing pandas display:")
print(df_tracks[['TrackId', 'Name', 'Composer', 'Milliseconds']])

#### Exercise 4: Albums from Tracks


In [ ]:
# Joined Track and Album tables using relationship
query = session.query(Track.Name, Album.Title).join(Album).limit(20)
track_album_data = pd.read_sql(query.statement, engine)

print("First 20 tracks with their album titles:")
print(track_album_data)

### Exercise 5: Tracks Sold


In [ ]:
# First 10 track sales
first_10_sales = session.query(InvoiceItem).limit(10).all()

print("First 10 track sales from invoice_items:")
for i, sale in enumerate(first_10_sales, 1):
    print(f"{i:2d}. InvoiceLineId: {sale.InvoiceLineId}, TrackId: {sale.TrackId}, "
          f"Quantity: {sale.Quantity}, UnitPrice: ${sale.UnitPrice}")

# Get track names and quantities for first 10 sales
query = session.query(Track.Name, InvoiceItem.Quantity, InvoiceItem.UnitPrice)\
    .join(InvoiceItem)\
    .limit(10)

sales_with_names = pd.read_sql(query.statement, engine)
print(f"\nTrack names and quantities sold (first 10):")
print(sales_with_names)

#### Exercise 6: Top Tracks Sold

In [ ]:
# Used aggregation with func.sum() and group_by()
top_tracks_query = session.query(
    Track.Name,
    func.sum(InvoiceItem.Quantity).label('total_sold')
)\
.join(InvoiceItem)\
.group_by(Track.TrackId, Track.Name)\
.order_by(func.sum(InvoiceItem.Quantity).desc())\
.limit(10)

top_tracks_df = pd.read_sql(top_tracks_query.statement, engine)

print("Top 10 most sold tracks:")
print(top_tracks_df)

print(f"\nDetailed breakdown:")
for i, row in top_tracks_df.iterrows():
    print(f"{i+1:2d}. '{row['Name']}' - Sold {row['total_sold']} times")

#### Exercise 7: Top Selling Artists


In [ ]:
#  4-table join: Artist → Album → Track → InvoiceItem
top_artists_query = session.query(
    Artist.Name,
    func.sum(InvoiceItem.Quantity).label('total_sales')
)\
.join(Album, Artist.ArtistId == Album.ArtistId)\
.join(Track, Album.AlbumId == Track.AlbumId)\
.join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId)\
.group_by(Artist.ArtistId, Artist.Name)\
.order_by(func.sum(InvoiceItem.Quantity).desc())\
.limit(10)

top_artists_df = pd.read_sql(top_artists_query.statement, engine)

print("Top 10 highest selling artists:")
print(top_artists_df)

print(f"\nDetailed ranking:")
for i, row in top_artists_df.iterrows():
    print(f"{i+1:2d}. {row['Name']} - {row['total_sales']} tracks sold")

# Close connections
cur.close()
session.close()
print(f"\n✅ All exercises completed successfully!")